In [8]:
import pandas as pd
import os
import numpy as np

base_path = r"c:\Users\User RPL 04\Downloads\DustiniaDelixia_Groceria-20260610T182424Z-3-001\DustiniaDelixia_Groceria"

customers = pd.read_csv(os.path.join(base_path, "customers.csv"))
orders = pd.read_csv(os.path.join(base_path, "orders.csv"))
order_items = pd.read_csv(os.path.join(base_path, "order_items.csv"))
payments = pd.read_csv(os.path.join(base_path, "order_payments.csv"))

In [9]:
# 1. Hitung total value per order
order_total = order_items.groupby("order_id")["price"].sum().reset_index()
order_total.rename(columns={"price":"order_total"}, inplace=True)

In [10]:
# 2. Gabungkan dengan tabel orders
orders_finance = orders.merge(order_total, on="order_id", how="left")

In [11]:
# 3. Gabungkan dengan tabel customers
customer_orders = orders_finance.merge(customers, on="customer_id", how="left")

# 4. Hitung total belanja per customer
customer_value = customer_orders.groupby("customer_id").agg(
    total_spend=("order_total","sum"),
    total_orders=("order_id","count")
).reset_index()

In [12]:
# 5. Tandai pelanggan High Value (top 10%)
threshold = customer_value["total_spend"].quantile(0.90)
customer_value["customer_segment"] = np.where(
    customer_value["total_spend"] >= threshold,
    "High Value",
    "Regular"
)

print("Jumlah High Value Customer:", (customer_value["customer_segment"]=="High Value").sum())

Jumlah High Value Customer: 10046


In [13]:
# 6. Gabungkan dengan payment untuk analisis metode pembayaran
finance_data = customer_orders[["order_id", "customer_id", "customer_city", "customer_state", "order_total"]].merge(
    payments,
    on="order_id",
    how="left"
)

# 7. Ringkasan metode pembayaran
payment_summary = finance_data.groupby("payment_type").agg(
    total_revenue=("payment_value","sum"),
    total_transactions=("order_id","count")
).reset_index().sort_values(by="total_revenue", ascending=False)

print("\nPayment Summary:")
print(payment_summary.head())


Payment Summary:
  payment_type  total_revenue  total_transactions
1  credit_card    12542084.19               76795
0       boleto     2869361.27               19784
4      voucher      379436.87                5775
2   debit_card      217989.79                1529
3  not_defined           0.00                   3


In [14]:
# 8. Ringkasan geografis revenue per state
geo_summary = finance_data.groupby("customer_state").agg(
    total_revenue=("payment_value","sum"),
    total_transactions=("order_id","count")
).reset_index().sort_values(by="total_revenue", ascending=False)

print("\nTop 10 States by Revenue:")
print(geo_summary.head(10))


Top 10 States by Revenue:
   customer_state  total_revenue  total_transactions
25             SP     5998226.96               43623
18             RJ     2144379.69               13527
10             MG     1872257.26               12102
22             RS      890898.54                5668
17             PR      811156.38                5262
23             SC      623086.43                3754
4              BA      616645.82                3610
6              DF      355141.08                2204
8              GO      350092.31                2112
7              ES      325967.55                2107


In [15]:
total_revenue = customer_value["total_spend"].sum()

high_value_revenue = customer_value[
    customer_value["customer_segment"]=="High Value"
]["total_spend"].sum()

print(f"Total Revenue: {total_revenue:,.2f}")
print(f"High Value Revenue: {high_value_revenue:,.2f}")
print(f"Contribution: {high_value_revenue/total_revenue*100:.2f}%")

Total Revenue: 13,591,643.70
High Value Revenue: 5,634,190.35
Contribution: 41.45%


In [16]:
finance_data = customer_orders.merge(
    payments,
    on="order_id",
    how="left"
)

finance_data = finance_data.merge(
    customer_value[["customer_id","customer_segment"]],
    on="customer_id",
    how="left"
)

In [17]:
payment_segment = finance_data.groupby(
    ["customer_segment","payment_type"]
).agg(
    total_revenue=("payment_value","sum"),
    total_transactions=("order_id","count")
).reset_index()

payment_segment

,customer_segment,payment_type,total_revenue,total_transactions
0,High Value,boleto,975430.41,1572
1,High Value,credit_card,4946631.30,8386
2,High Value,debit_card,73751.20,111
3,High Value,voucher,88025.13,447
4,Regular,boleto,1893930.86,18212
5,Regular,credit_card,7595452.89,68409
6,Regular,debit_card,144238.59,1418
7,Regular,not_defined,0.00,3
8,Regular,voucher,291411.74,5328


In [18]:
installment_analysis = finance_data.groupby(
    "payment_installments"
).agg(
    avg_payment=("payment_value","mean"),
    total_transactions=("order_id","count")
).reset_index()

installment_analysis.sort_values(
    "payment_installments"
)

,payment_installments,avg_payment,total_transactions
0,0.0,94.315000,2
1,1.0,112.420229,52546
2,2.0,127.228150,12413
3,3.0,142.539317,10461
4,4.0,163.976840,7098
5,5.0,183.465222,5239
6,6.0,209.849952,3920
7,7.0,187.673672,1626
8,8.0,307.737427,4268
9,9.0,203.440870,644


In [19]:
high_value_geo = finance_data[
    finance_data["customer_segment"]=="High Value"
]

state_summary = high_value_geo.groupby(
    "customer_state"
).agg(
    revenue=("payment_value","sum"),
    transactions=("order_id","count")
).reset_index()

state_summary.sort_values(
    "revenue",
    ascending=False
).head(10)

,customer_state,revenue,transactions
25,SP,2093759.55,3825
18,RJ,804054.12,1355
10,MG,675749.97,1151
22,RS,333291.36,596
17,PR,314290.03,535
4,BA,258561.99,418
23,SC,242266.32,403
6,DF,140113.21,228
8,GO,139868.17,236
5,CE,138762.74,246


In [20]:
total_revenue = customer_value["total_spend"].sum()

high_value_revenue = customer_value[
    customer_value["customer_segment"]=="High Value"
]["total_spend"].sum()

print(f"Contribution: {high_value_revenue/total_revenue*100:.2f}%")

Contribution: 41.45%
